# StandUp4AI: 1000-Video Evaluation Pipeline

Uses partition.csv for ALL labels (370 val + 3281 train). 

In [ ]:
# Cell 1: Setup
BASE = "/content/drive/MyDrive/standup4ai"
AUDIO_DIR = f"{BASE}/audio"
LABELS_DIR = f"{BASE}/labels"
OUT_DIR = f"{BASE}/eval_1000"
PARTITION = f"{BASE}/standup4ai_partition.csv"

import os, subprocess, json
os.makedirs(OUT_DIR, exist_ok=True)

print(f"Audio: {AUDIO_DIR}")
print(f"Partition: {PARTITION}")
print(f"Output: {OUT_DIR}")

# Check Drive contents
r = subprocess.run(['rclone', 'ls', AUDIO_DIR], capture_output=True, text=True, timeout=120)
drive_files = set()
for ln in r.stdout.splitlines():
    if '.m4a' in ln:
        vid = ln.split()[-1].replace('.m4a','')
        drive_files.add(vid)
print(f"Videos on Drive: {len(drive_files)}")

In [ ]:
# Cell 2: Load partition + labels
# Partition CSV: fn,lan,part
val_ids = set()
train_ids = set()
with open(PARTITION) as f:
    next(f)  # header
    for ln in f:
        p = ln.strip().split(',')
        if len(p) >= 3:
            vid, lang, part = p[0], p[1], p[2]
            if part == 'val': val_ids.add(vid)
            elif part == 'train': train_ids.add(vid)

print(f"Val: {len(val_ids)}, Train: {len(train_ids)}")

# Load ALL labels from partition file
# Format: fn,lan,part + later lines: t0,t1,source,label
# For each video, we have multiple time ranges with risa/no_risa labels
label_data = {}  # vid -> list of (t0, t1, label)

# First pass: collect video partitions
vid_part = {}  # vid -> part
with open(PARTITION) as f:
    next(f)
    for ln in f:
        p = ln.strip().split(',')
        if len(p) >= 3:
            vid_part[p[0]] = p[2]

# Labels are stored as separate CSV files per video
# But we can also extract from partition file pattern:
# Actually partition only has fn,lan,part - not labels
# Labels are in individual files like /labels/{vid}.csv

# Check which label files exist
r2 = subprocess.run(['rclone', 'ls', LABELS_DIR], capture_output=True, text=True, timeout=120)
label_files = set()
for ln in r2.stdout.splitlines():
    if '.csv' in ln:
        vid = ln.split()[-1].replace('.csv','')
        label_files.add(vid)

print(f"Label files: {len(label_files)}")

# Eval-ready = videos with BOTH audio AND labels
val_eval = val_ids & drive_files & label_files
train_eval = train_ids & drive_files & label_files
print(f"Val eval-ready: {len(val_eval)}")
print(f"Train eval-ready: {len(train_eval)}")
print(f"Total eval-ready: {len(val_eval) + len(train_eval)}")

In [ ]:
# Cell 3: Feature extraction
import numpy as np
import librosa
from tqdm import tqdm

def extract_features(audio_path, sr=22050):
    """Extract 20-dim spectral features."""
    try:
        y, sr = librosa.load(audio_path, sr=sr, mono=True)
    except:
        return None
    
    features = []
    hop_length = 512
    
    # RMS energy
    rms = librosa.feature.rms(y=y, hop_length=hop_length)[0]
    
    # ZCR
    zcr = librosa.feature.zero_crossing_rate(y, hop_length=hop_length)[0]
    
    # Spectral centroid
    spec_cent = librosa.feature.spectral_centroid(y=y, sr=sr, hop_length=hop_length)[0]
    
    # Spectral bandwidth
    spec_bw = librosa.feature.spectral_bandwidth(y=y, sr=sr, hop_length=hop_length)[0]
    
    # Spectral rolloff
    spec_roll = librosa.feature.spectral_rolloff(y=y, sr=sr, hop_length=hop_length)[0]
    
    # Spectral flatness
    spec_flat = librosa.feature.spectral_flatness(y=y, hop_length=hop_length)[0]
    
    # MFCCs (13 dim)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13, hop_length=hop_length)[0]
    
    # Delta MFCCs (13 dim)
    mfcc_delta = librosa.feature.delta(mfcc)
    
    # Pad to same length
    max_len = max(len(rms), len(zcr), len(spec_cent))
    
    def pad(x):
        if len(x) < max_len:
            x = np.pad(x, (0, max_len - len(x)))
        return x[:max_len]
    
    feat = np.stack([pad(rms), pad(zcr), pad(spec_cent), pad(spec_bw),
                    pad(spec_roll), pad(spec_flat)] + 
                   [pad(m) for m in mfcc] + [pad(m) for m in mfcc_delta])
    
    return feat.T  # (n_frames, 20)

In [ ]:
# Cell 4: Extract features for eval-ready videos
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import f1_score, precision_score, recall_score

# Combine val + train for 1000-video eval
all_eval = list(val_eval | train_eval)[:1000]  # cap at 1000
print(f"Evaluating on {len(all_eval)} videos")

# Extract features + labels per video
X_all, y_all, info_all = [], [], []

for vid in tqdm(all_eval, desc="Extracting"):
    audio_path = f"{AUDIO_DIR}/{vid}.m4a"
    label_path = f"{LABELS_DIR}/{vid}.csv"
    
    # Check audio exists locally (may need download)
    if not os.path.exists(audio_path):
        # Try to download from Drive
        r = subprocess.run(['rclone', 'copy', audio_path, '/tmp/'], 
                        capture_output=True, text=True, timeout=60)
        if r.returncode != 0 or not os.path.exists(f"/tmp/{vid}.m4a"):
            continue
        audio_path = f"/tmp/{vid}.m4a"
    
    # Load labels
    labels = []
    try:
        with open(label_path) as f:
            next(f)  # header
            for ln in f:
                p = ln.strip().split(',')
                if len(p) >= 4:
                    t0, t1, source, label = float(p[0]), float(p[1]), p[2], p[3].strip()
                    is_laugh = 1 if label == 'risa' else 0
                    labels.append((t0, t1, is_laugh))
    except:
        continue
    
    if not labels:
        continue
    
    # Extract features
    feat = extract_features(audio_path)
    if feat is None:
        continue
    
    # Aggregate per-label segment
    for t0, t1, label in labels:
        start_frame = int(t0 * 22050 / 512)
        end_frame = int(t1 * 22050 / 512)
        if end_frame <= start_frame or start_frame >= len(feat):
            continue
        seg_feat = feat[start_frame:min(end_frame, len(feat))]
        if len(seg_feat) < 5:
            continue
        X_all.append(seg_feat.mean(axis=0))
        y_all.append(label)
        info_all.append(vid)

X_all = np.array(X_all)
y_all = np.array(y_all)
print(f"Total samples: {len(X_all)}, Pos rate: {y_all.mean():.3f}")
print(f"Shape: {X_all.shape}")

In [ ]:
# Cell 5: Train + Evaluate
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_all)

# 5-fold CV
models = {
    'LogReg': LogisticRegression(max_iter=500, C=1.0, class_weight='balanced'),
    'RF': RandomForestClassifier(n_estimators=100, max_depth=10, class_weight='balanced', n_jobs=-1)
}

results = {}
for name, model in models.items():
    y_pred = cross_val_predict(model, X_scaled, y_all, cv=5, n_jobs=-1)
    f1 = f1_score(y_all, y_pred)
    prec = precision_score(y_all, y_pred)
    rec = recall_score(y_all, y_pred)
    results[name] = {'f1': f1, 'precision': prec, 'recall': rec}
    print(f"{name}: F1={f1:.4f} P={prec:.4f} R={rec:.4f}")

# Best model
best = max(results.items(), key=lambda x: x[1]['f1'])
print(f"\nBest: {best[0]} F1={best[1]['f1']:.4f}")

In [ ]:
# Cell 6: Save results
out = {
    'n_samples': len(X_all),
    'n_videos': len(all_eval),
    'positive_rate': float(y_all.mean()),
    'feature_dim': X_all.shape[1],
    'results': results,
    'baseline_f1': 0.51  # StandUp4AI baseline
}

import json
out_path = f"{OUT_DIR}/eval_1000_results.json"
with open(out_path, 'w') as f:
    json.dump(results, f, indent=2)
print(f"Saved to {out_path}")

# Also save features for later use
np.savez(f"{OUT_DIR}/eval_1000_features.npz", X=X_all, y=y_all, info=info_all)
print(f"Features saved: {X_all.shape}")